In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score
import warnings
warnings.filterwarnings('ignore')

# Список файлов
files = {
    'qwen_tqa_cut.csv': 'Qwen_TruthfulQA',
    'qwen_slava_cut.csv': 'Qwen_Slava',
    'yandex_gpt_slava_answered.csv': 'YandexGPT_Slava',
    'yandex_gpt_tqa_answered.csv': 'YandexGPT_TruthfulQA',
    'gpt_slava_annotated.csv': 'ChatGPT_Slava',
    'gpt_tqa_answered.csv': 'ChatGPT_TruthfulQA',
    'giga_slava_annotated.csv': 'GigaChat_Slava',
    'giga_tqa_answered.csv': 'GigaChat_TruthfulQA'
}

# Метрики и их направление
metrics_direction = {
    'Semantic Entropy': 'higher',
    'EigenScore': 'higher',
    'SAR Score': 'lower',
    'NumSets': 'higher',
    'Self Confidence': 'lower',
    'Verbalized Uncertainty': 'lower'
}

all_data = []
for file, name in files.items():
    try:
        df = pd.read_csv(file, encoding='utf-8')
        df['source'] = name
        all_data.append(df)
        print(f"✅ Loaded {file}: {len(df)} rows")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

df_all = pd.concat(all_data, ignore_index=True)
df_all['target'] = df_all['ground_truth']
df_all['model'] = df_all['source'].apply(lambda x: x.split('_')[0])
df_all['dataset'] = df_all['source'].apply(lambda x: x.split('_')[1])

print(f"\n📊 Total: {len(df_all)} rows")
available_metrics = [m for m in metrics_direction.keys() if m in df_all.columns]
print(f"Available metrics: {available_metrics}")

✅ Loaded qwen_tqa_cut.csv: 790 rows
✅ Loaded qwen_slava_cut.csv: 500 rows
✅ Loaded yandex_gpt_slava_answered.csv: 481 rows
✅ Loaded yandex_gpt_tqa_answered.csv: 755 rows
✅ Loaded gpt_slava_annotated.csv: 500 rows
✅ Loaded gpt_tqa_answered.csv: 785 rows
✅ Loaded giga_slava_annotated.csv: 500 rows
✅ Loaded giga_tqa_answered.csv: 761 rows

📊 Total: 5072 rows
Available metrics: ['Semantic Entropy', 'EigenScore', 'SAR Score', 'NumSets', 'Self Confidence', 'Verbalized Uncertainty']


In [ ]:
def find_best_threshold_for_metric(values, target, direction, scoring='f1'):

    best_score = 0
    best_thresh = 0.5

    for thresh in np.linspace(values.min(), values.max(), 100):
        if direction == 'higher':
            pred = (values > thresh).astype(int)
        else:
            pred = (values < thresh).astype(int)

        if scoring == 'f1':
            score = f1_score(target, pred, zero_division=0)
        else:
            score = recall_score(target, pred, zero_division=0)

        if score > best_score:
            best_score = score
            best_thresh = thresh

    return best_thresh, best_score


def evaluate_metric_with_cv(df, metric_name, direction, cv_folds=5, scoring='f1'):

    X = df[metric_name].values
    y = df['target'].values
    valid_idx = ~np.isnan(X)
    X = X[valid_idx]
    y = y[valid_idx]

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    f1_scores = []
    recall_scores = []
    precision_scores = []
    thresholds = []

    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        best_thresh, _ = find_best_threshold_for_metric(
            pd.Series(X_train), pd.Series(y_train),
            direction, scoring=scoring
        )
        thresholds.append(best_thresh)

        if direction == 'higher':
            y_pred = (X_test > best_thresh).astype(int)
        else:
            y_pred = (X_test < best_thresh).astype(int)

        f1_scores.append(f1_score(y_test, y_pred, zero_division=0))
        recall_scores.append(recall_score(y_test, y_pred, zero_division=0))
        precision_scores.append(precision_score(y_test, y_pred, zero_division=0))

    return {
        'f1_mean': np.mean(f1_scores),
        'f1_std': np.std(f1_scores),
        'recall_mean': np.mean(recall_scores),
        'recall_std': np.std(recall_scores),
        'precision_mean': np.mean(precision_scores),
        'precision_std': np.std(precision_scores),
        'threshold_mean': np.mean(thresholds),
        'threshold_std': np.std(thresholds)
    }

In [ ]:
def evaluate_voting_with_cv(df, metrics_direction, cv_folds=5, scoring='f1'):

    metric_names = [m for m in metrics_direction.keys() if m in df.columns]

    X_list = []
    for metric in metric_names:
        values = df[metric].values
        X_list.append(values)

    X = np.column_stack(X_list)
    y = df['target'].values
    valid_idx = ~np.isnan(X).any(axis=1)
    X = X[valid_idx]
    y = y[valid_idx]

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    f1_scores = []
    recall_scores = []
    precision_scores = []
    thresholds = []

    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        train_votes = []
        for i in range(len(X_train)):
            votes = 0
            for j, metric in enumerate(metric_names):
                direction = metrics_direction[metric]

                val = X_train[i, j]

                pass

        thresholds_per_metric = []
        for j, metric in enumerate(metric_names):
            direction = metrics_direction[metric]
            best_thresh, _ = find_best_threshold_for_metric(
                pd.Series(X_train[:, j]), pd.Series(y_train),
                direction, scoring=scoring
            )
            thresholds_per_metric.append(best_thresh)

        train_votes = np.zeros(len(X_train))
        for j, metric in enumerate(metric_names):
            direction = metrics_direction[metric]
            thresh = thresholds_per_metric[j]
            if direction == 'higher':
                train_votes += (X_train[:, j] > thresh).astype(int)
            else:
                train_votes += (X_train[:, j] < thresh).astype(int)

        train_votes = train_votes / len(metric_names)

        best_vote_thresh, _ = find_best_threshold_for_metric(
            pd.Series(train_votes), pd.Series(y_train),
            'higher', scoring=scoring
        )
        thresholds.append(best_vote_thresh)

        test_votes = np.zeros(len(X_test))
        for j, metric in enumerate(metric_names):
            direction = metrics_direction[metric]
            thresh = thresholds_per_metric[j]
            if direction == 'higher':
                test_votes += (X_test[:, j] > thresh).astype(int)
            else:
                test_votes += (X_test[:, j] < thresh).astype(int)

        test_votes = test_votes / len(metric_names)
        y_pred = (test_votes > best_vote_thresh).astype(int)

        f1_scores.append(f1_score(y_test, y_pred, zero_division=0))
        recall_scores.append(recall_score(y_test, y_pred, zero_division=0))
        precision_scores.append(precision_score(y_test, y_pred, zero_division=0))

    return {
        'f1_mean': np.mean(f1_scores),
        'f1_std': np.std(f1_scores),
        'recall_mean': np.mean(recall_scores),
        'recall_std': np.std(recall_scores),
        'precision_mean': np.mean(precision_scores),
        'precision_std': np.std(precision_scores),
        'threshold_mean': np.mean(thresholds),
        'threshold_std': np.std(thresholds)
    }

In [ ]:
#ЗАПУСК ДЛЯ ВСЕХ СРЕЗОВ (ОПТИМИЗАЦИЯ ПО F1)

print("="*80)
print("ОПТИМИЗАЦИЯ ПО F1-SCORE")
print("="*80)

all_slices = []

for name in files.values():
    all_slices.append(name)
all_slices.append('ALL_TruthfulQA')
all_slices.append('ALL_Slava')
for model_name in ['Qwen', 'YandexGPT', 'ChatGPT', 'GigaChat']:
    all_slices.append(f'ALL_{model_name}')
all_slices.append('FULL_DATASET')


results_f1 = []

for slice_name in all_slices:
    print(f"\n📁 Processing: {slice_name}")

    # Получаем данные для среза
    if slice_name in files.values():
        file = [f for f, name in files.items() if name == slice_name][0]
        df = pd.read_csv(file, encoding='utf-8')
        df['target'] = df['ground_truth']
    elif slice_name.startswith('ALL_TruthfulQA'):
        df = df_all[df_all['dataset'] == 'TruthfulQA'].copy()
    elif slice_name.startswith('ALL_Slava'):
        df = df_all[df_all['dataset'] == 'Slava'].copy()
    elif slice_name.startswith('ALL_Qwen'):
        df = df_all[df_all['model'] == 'Qwen'].copy()
    elif slice_name.startswith('ALL_YandexGPT'):
        df = df_all[df_all['model'] == 'YandexGPT'].copy()
    elif slice_name.startswith('ALL_ChatGPT'):
        df = df_all[df_all['model'] == 'ChatGPT'].copy()
    elif slice_name.startswith('ALL_GigaChat'):
        df = df_all[df_all['model'] == 'GigaChat'].copy()
    elif slice_name == 'FULL_DATASET':
        df = df_all.copy()
    else:
        continue

    # Оцениваем каждую метрику
    for metric, direction in metrics_direction.items():
        if metric not in df.columns:
            continue

        result = evaluate_metric_with_cv(df, metric, direction, cv_folds=5, scoring='f1')
        results_f1.append({
            'slice': slice_name,
            'method': metric,
            'f1': result['f1_mean'],
            'f1_std': result['f1_std'],
            'recall': result['recall_mean'],
            'recall_std': result['recall_std'],
            'precision': result['precision_mean'],
            'precision_std': result['precision_std'],
            'threshold': result['threshold_mean'],
            'threshold_std': result['threshold_std']
        })

    # Оцениваем голосование
    if len([m for m in metrics_direction.keys() if m in df.columns]) >= 3:
        result_vote = evaluate_voting_with_cv(df, metrics_direction, cv_folds=5, scoring='f1')
        results_f1.append({
            'slice': slice_name,
            'method': 'Voting (6 metrics)',
            'f1': result_vote['f1_mean'],
            'f1_std': result_vote['f1_std'],
            'recall': result_vote['recall_mean'],
            'recall_std': result_vote['recall_std'],
            'precision': result_vote['precision_mean'],
            'precision_std': result_vote['precision_std'],
            'threshold': result_vote['threshold_mean'],
            'threshold_std': result_vote['threshold_std']
        })

df_results_f1 = pd.DataFrame(results_f1)
print("\n✅ F1 optimization completed")

ОПТИМИЗАЦИЯ ПО F1-SCORE

📁 Processing: Qwen_TruthfulQA

📁 Processing: Qwen_Slava

📁 Processing: YandexGPT_Slava

📁 Processing: YandexGPT_TruthfulQA

📁 Processing: ChatGPT_Slava

📁 Processing: ChatGPT_TruthfulQA

📁 Processing: GigaChat_Slava

📁 Processing: GigaChat_TruthfulQA

📁 Processing: ALL_TruthfulQA

📁 Processing: ALL_Slava

📁 Processing: ALL_Qwen

📁 Processing: ALL_YandexGPT

📁 Processing: ALL_ChatGPT

📁 Processing: ALL_GigaChat

📁 Processing: FULL_DATASET

✅ F1 optimization completed


In [ ]:
#ЗАПУСК ДЛЯ ВСЕХ СРЕЗОВ (ОПТИМИЗАЦИЯ ПО RECALL)

print("="*80)
print("ОПТИМИЗАЦИЯ ПО RECALL")
print("="*80)

results_recall = []

for slice_name in all_slices:
    print(f"\n📁 Processing: {slice_name}")

    # Получаем данные для среза
    if slice_name in files.values():
        file = [f for f, name in files.items() if name == slice_name][0]
        df = pd.read_csv(file, encoding='utf-8')
        df['target'] = df['ground_truth']
    elif slice_name.startswith('ALL_TruthfulQA'):
        df = df_all[df_all['dataset'] == 'TruthfulQA'].copy()
    elif slice_name.startswith('ALL_Slava'):
        df = df_all[df_all['dataset'] == 'Slava'].copy()
    elif slice_name.startswith('ALL_Qwen'):
        df = df_all[df_all['model'] == 'Qwen'].copy()
    elif slice_name.startswith('ALL_YandexGPT'):
        df = df_all[df_all['model'] == 'YandexGPT'].copy()
    elif slice_name.startswith('ALL_ChatGPT'):
        df = df_all[df_all['model'] == 'ChatGPT'].copy()
    elif slice_name.startswith('ALL_GigaChat'):
        df = df_all[df_all['model'] == 'GigaChat'].copy()
    elif slice_name == 'FULL_DATASET':
        df = df_all.copy()
    else:
        continue

    # Оцениваем каждую метрику
    for metric, direction in metrics_direction.items():
        if metric not in df.columns:
            continue

        result = evaluate_metric_with_cv(df, metric, direction, cv_folds=5, scoring='recall')
        results_recall.append({
            'slice': slice_name,
            'method': metric,
            'f1': result['f1_mean'],
            'f1_std': result['f1_std'],
            'recall': result['recall_mean'],
            'recall_std': result['recall_std'],
            'precision': result['precision_mean'],
            'precision_std': result['precision_std'],
            'threshold': result['threshold_mean'],
            'threshold_std': result['threshold_std']
        })

    # Оцениваем голосование
    if len([m for m in metrics_direction.keys() if m in df.columns]) >= 3:
        result_vote = evaluate_voting_with_cv(df, metrics_direction, cv_folds=5, scoring='recall')
        results_recall.append({
            'slice': slice_name,
            'method': 'Voting (6 metrics)',
            'f1': result_vote['f1_mean'],
            'f1_std': result_vote['f1_std'],
            'recall': result_vote['recall_mean'],
            'recall_std': result_vote['recall_std'],
            'precision': result_vote['precision_mean'],
            'precision_std': result_vote['precision_std'],
            'threshold': result_vote['threshold_mean'],
            'threshold_std': result_vote['threshold_std']
        })

df_results_recall = pd.DataFrame(results_recall)
print("\n✅ Recall optimization completed")

ОПТИМИЗАЦИЯ ПО RECALL

📁 Processing: Qwen_TruthfulQA

📁 Processing: Qwen_Slava

📁 Processing: YandexGPT_Slava

📁 Processing: YandexGPT_TruthfulQA

📁 Processing: ChatGPT_Slava

📁 Processing: ChatGPT_TruthfulQA

📁 Processing: GigaChat_Slava

📁 Processing: GigaChat_TruthfulQA

📁 Processing: ALL_TruthfulQA

📁 Processing: ALL_Slava

📁 Processing: ALL_Qwen

📁 Processing: ALL_YandexGPT

📁 Processing: ALL_ChatGPT

📁 Processing: ALL_GigaChat

📁 Processing: FULL_DATASET

✅ Recall optimization completed


In [ ]:

# Results

print("="*80)
print("ПОЛНЫЕ РЕЗУЛЬТАТЫ: ОПТИМИЗАЦИЯ ПО F1")
print("="*80)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Полная таблица по F1 оптимизации
df_results_f1_full = df_results_f1[['slice', 'method', 'f1', 'recall', 'precision', 'threshold']].copy()
df_results_f1_full = df_results_f1_full.round(4)
df_results_f1_full = df_results_f1_full.sort_values(['slice', 'f1'], ascending=[True, False])
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(df_results_f1_full.to_string())

df_results_f1_full.to_csv('threshold_optimization_f1_full.csv', index=False)

print("\n" + "="*80)
print("ПОЛНЫЕ РЕЗУЛЬТАТЫ: ОПТИМИЗАЦИЯ ПО RECALL")
print("="*80)

# Полная таблица по Recall оптимизации
df_results_recall_full = df_results_recall[['slice', 'method', 'recall', 'f1', 'precision', 'threshold']].copy()
df_results_recall_full = df_results_recall_full.round(4)
df_results_recall_full = df_results_recall_full.sort_values(['slice', 'recall'], ascending=[True, False])

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(df_results_recall_full.to_string())

df_results_recall_full.to_csv('threshold_optimization_recall_full.csv', index=False)

print("\n Full results saved to CSV files")
print("   - threshold_optimization_f1_full.csv")
print("   - threshold_optimization_recall_full.csv")

ПОЛНЫЕ РЕЗУЛЬТАТЫ: ОПТИМИЗАЦИЯ ПО F1
                    slice                  method      f1  recall  precision  threshold
89            ALL_ChatGPT  Verbalized Uncertainty  0.3368  0.3721     0.3077     0.7071
90            ALL_ChatGPT      Voting (6 metrics)  0.2576  0.3953     0.1910     0.1717
85            ALL_ChatGPT              EigenScore  0.2456  0.3256     0.1972     0.3093
88            ALL_ChatGPT         Self Confidence  0.2388  0.1860     0.3333     0.6300
84            ALL_ChatGPT        Semantic Entropy  0.2203  0.3023     0.1733     0.1477
86            ALL_ChatGPT               SAR Score  0.2182  0.2791     0.1791     0.8689
87            ALL_ChatGPT                 NumSets  0.2136  0.2558     0.1833     1.0000
92           ALL_GigaChat              EigenScore  0.2870  0.6154     0.1871     0.2309
97           ALL_GigaChat      Voting (6 metrics)  0.2832  0.6154     0.1839     0.1717
95           ALL_GigaChat         Self Confidence  0.2564  0.5769     0.1648     0.

In [ ]:
#ЛУЧШИЙ МЕТОД ДЛЯ КАЖДОГО СРЕЗА

print("="*80)
print("ЛУЧШИЙ МЕТОД ДЛЯ КАЖДОГО СРЕЗА")
print("="*80)

# Лучший по F1 для каждого среза
best_f1_per_slice = df_results_f1.loc[df_results_f1.groupby('slice')['f1'].idxmax()]
best_f1_per_slice = best_f1_per_slice[['slice', 'method', 'f1', 'recall', 'precision', 'threshold']].round(4)
best_f1_per_slice = best_f1_per_slice.sort_values('f1', ascending=False)
print("\n ЛУЧШИЙ ПО F1:")
display(best_f1_per_slice)

# Лучший по Recall для каждого среза
best_recall_per_slice = df_results_recall.loc[df_results_recall.groupby('slice')['recall'].idxmax()]
best_recall_per_slice = best_recall_per_slice[['slice', 'method', 'recall', 'f1', 'precision', 'threshold']].round(4)
best_recall_per_slice = best_recall_per_slice.sort_values('recall', ascending=False)
print("\n ЛУЧШИЙ ПО RECALL:")
display(best_recall_per_slice)

best_f1_per_slice.to_csv('threshold_optimization_best_f1.csv', index=False)
best_recall_per_slice.to_csv('threshold_optimization_best_recall.csv', index=False)

ЛУЧШИЙ МЕТОД ДЛЯ КАЖДОГО СРЕЗА

 ЛУЧШИЙ ПО F1:


,slice,method,f1,recall,precision,threshold
9,Qwen_Slava,SAR Score,0.6789,0.9487,0.5286,0.9224
71,ALL_Qwen,EigenScore,0.6382,0.9609,0.4778,0.2290
1,Qwen_TruthfulQA,EigenScore,0.6113,0.8020,0.4939,0.3675
69,ALL_Slava,Voting (6 metrics),0.5424,0.6957,0.4444,0.3333
39,ChatGPT_TruthfulQA,Self Confidence,0.5000,0.4194,0.6190,0.6300
104,FULL_DATASET,Voting (6 metrics),0.4735,0.6751,0.3646,0.1717
62,ALL_TruthfulQA,Voting (6 metrics),0.4265,0.6535,0.3165,0.1717
33,ChatGPT_Slava,Verbalized Uncertainty,0.4211,0.3333,0.5714,0.7071
46,GigaChat_Slava,Self Confidence,0.3810,0.3077,0.5000,0.9515
81,ALL_YandexGPT,Self Confidence,0.3373,0.3256,0.3500,0.8586



 ЛУЧШИЙ ПО RECALL:


,slice,method,recall,f1,precision,threshold
84,ALL_ChatGPT,Semantic Entropy,1.0000,0.2005,0.1114,-0.0000
70,ALL_Qwen,Semantic Entropy,1.0000,0.6325,0.4625,0.0000
63,ALL_Slava,Semantic Entropy,1.0000,0.3239,0.1933,-0.0000
78,ALL_YandexGPT,EigenScore,1.0000,0.2113,0.1181,0.0000
56,ALL_TruthfulQA,Semantic Entropy,1.0000,0.3578,0.2179,-0.0000
28,ChatGPT_Slava,Semantic Entropy,1.0000,0.1481,0.0800,-0.0000
35,ChatGPT_TruthfulQA,Semantic Entropy,1.0000,0.2348,0.1330,-0.0000
7,Qwen_Slava,Semantic Entropy,1.0000,0.6842,0.5200,0.0349
98,FULL_DATASET,Semantic Entropy,1.0000,0.3448,0.2083,-0.0000
42,GigaChat_Slava,Semantic Entropy,1.0000,0.1595,0.0867,-0.0000
